# Notebook 10: Backend Data Export (Store CA_1)
## Step 1: Import Libraries and Inspect Uploaded Files

### What we are doing in this cell
We are importing the Python libraries needed for data export and verification. Then, we list every file currently uploaded in `/content/` along with its file size, and check which of our required backend handover files are already present.

### Why we need this step
Before creating `history_small.parquet`, `calendar_small.csv`, and `features.py`, we must inspect the exact filenames of our uploaded historical sales data, calendar file, and previously exported models/tables so we do not assume any file names or paths.

### What problem this step solves
It prevents `FileNotFoundError` crashes and gives us a clear starting inventory of what is already ready vs. what we need to generate in Notebook 10.

### What output we should expect
A complete list of all files uploaded to `/content/` with their sizes, followed by a status checklist (`[OK]` or `[MISSING]`) for our known backend files.

In [ ]:
# Standard Python libraries
import os
import json
import warnings

# Data and model libraries
import numpy as np
import pandas as pd
import joblib

# Keep output clean
warnings.filterwarnings("ignore")

# Local folder where we will organize all final backend handover files
HANDOVER_DIR = "/content/backend_handover"
os.makedirs(HANDOVER_DIR, exist_ok=True)

# 1. List ALL files currently uploaded in /content/ (excluding Colab's default sample_data)
print("--- All Files Currently in /content/ ---")
uploaded_files = sorted([
    f for f in os.listdir("/content")
    if os.path.isfile(os.path.join("/content", f)) and not f.startswith(".")
])

for filename in uploaded_files:
    filepath = os.path.join("/content", filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f" - {filename:<35} ({size_mb:.2f} MB)")

# 2. Check our known previously exported files
expected_existing_files = [
    "lightgbm_h7.joblib",
    "lightgbm_h28.joblib",
    "feature_lists.json",
    "anomalies_validation.parquet",
    "item_groups.csv",
    "group_thresholds.csv",
    "item_error_spread.csv",
    "reorder.py",
    "ca1_h7_val.parquet",
    "ca1_h28_val.parquet"
]

print("\n--- Checking Known Exported Files ---")
for fname in expected_existing_files:
    in_content = os.path.exists(os.path.join("/content", fname))
    in_handover = os.path.exists(os.path.join(HANDOVER_DIR, fname))

    if in_content:
        print(f"[OK]      {fname:<32} (found in /content/)")
    elif in_handover:
        print(f"[OK]      {fname:<32} (found in /content/backend_handover/)")
    else:
        print(f"[MISSING] {fname:<32} (not found)")

--- All Files Currently in /content/ ---
 - CA_1_clean.parquet                  (9.24 MB)
 - anomalies_validation.parquet        (0.24 MB)
 - ca1_h28_val.parquet                 (0.23 MB)
 - ca1_h7_train.parquet                (14.83 MB)
 - ca1_h7_val.parquet                  (0.22 MB)
 - calendar.csv                        (0.10 MB)
 - feature_lists.json                  (0.06 MB)
 - group_thresholds.csv                (0.00 MB)
 - h7_val_predictions.parquet          (0.78 MB)
 - item_error_spread.csv               (0.22 MB)
 - item_groups.csv                     (0.30 MB)
 - lightgbm_h28.joblib                 (8.33 MB)
 - lightgbm_h7.joblib                  (1.74 MB)
 - reorder.py                          (0.00 MB)

--- Checking Known Exported Files ---
[OK]      lightgbm_h7.joblib               (found in /content/)
[OK]      lightgbm_h28.joblib              (found in /content/)
[OK]      feature_lists.json               (found in /content/)
[OK]      anomalies_validation.parquet   

## Step 2: Inspect Source Data (`CA_1_clean.parquet`, `calendar.csv`, and `feature_lists.json`)

### What we are doing in this cell
We are loading `CA_1_clean.parquet`, `calendar.csv`, and `feature_lists.json` to inspect their exact column names, data types, shapes, and date ranges. We also check the exact feature lists (`features_7` and `features_28`) required by our LightGBM models.

### Why we need this step
Before creating `history_small.parquet`, `calendar_small.csv`, and `features.py`, we must know the exact column names and date boundaries in our source files. Checking the maximum date in `CA_1_clean.parquet` is especially important so we do not accidentally include untouched test dates (`2016-04-25` to `2016-05-22`).

### What problem this step solves
It prevents `KeyError` bugs from guessing column names and ensures zero test-data leakage when we slice the last 200 days of history.

### What output we should expect
The shapes, column lists, date ranges, and first 3 rows of `CA_1_clean.parquet` and `calendar.csv`, plus the exact feature lists from `feature_lists.json`.

In [ ]:
# 1. Load source historical data, calendar, and feature_lists.json
CA1_CLEAN_PATH = "/content/CA_1_clean.parquet"
CALENDAR_PATH = "/content/calendar.csv"
FEATURES_JSON_PATH = "/content/feature_lists.json"

ca1_clean = pd.read_parquet(CA1_CLEAN_PATH)
calendar_df = pd.read_csv(CALENDAR_PATH)

with open(FEATURES_JSON_PATH, "r") as f:
    feature_lists = json.load(f)

# 2. Inspect CA_1_clean.parquet
print("--- CA_1_clean.parquet ---")
print(f"Shape:      {ca1_clean.shape[0]:,} rows, {ca1_clean.shape[1]} columns")
print(f"Columns:    {list(ca1_clean.columns)}")
if "date" in ca1_clean.columns:
    print(f"Date Range: {ca1_clean['date'].min()} to {ca1_clean['date'].max()}")
print("\nPreview (First 3 rows):")
print(ca1_clean.head(3))

# 3. Inspect calendar.csv
print("\n--- calendar.csv ---")
print(f"Shape:      {calendar_df.shape[0]:,} rows, {calendar_df.shape[1]} columns")
print(f"Columns:    {list(calendar_df.columns)}")
if "date" in calendar_df.columns:
    print(f"Date Range: {calendar_df['date'].min()} to {calendar_df['date'].max()}")
print("\nPreview (First 3 rows):")
print(calendar_df.head(3))

# 4. Inspect feature_lists.json
print("\n--- feature_lists.json ---")
print(f"features_7  ({len(feature_lists['features_7'])}):  {feature_lists['features_7']}")
print(f"features_28 ({len(feature_lists['features_28'])}): {feature_lists['features_28']}")
print(f"item_categories count: {len(feature_lists['item_categories']):,}")

--- CA_1_clean.parquet ---
Shape:      4,781,564 rows, 13 columns
Columns:    ['item_id', 'dept_id', 'cat_id', 'store_id', 'sales', 'date', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap', 'sell_price']
Date Range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00

Preview (First 3 rows):
       item_id  dept_id cat_id store_id  sales       date  wday  month  year  \
0  FOODS_1_001  FOODS_1  FOODS     CA_1      3 2011-01-29     1      1  2011   
1  FOODS_1_001  FOODS_1  FOODS     CA_1      0 2011-01-30     2      1  2011   
2  FOODS_1_001  FOODS_1  FOODS     CA_1      0 2011-01-31     3      1  2011   

  event_name_1 event_type_1  snap  sell_price  
0          NaN          NaN     0         2.0  
1          NaN          NaN     0         2.0  
2          NaN          NaN     0         2.0  

--- calendar.csv ---
Shape:      1,969 rows, 14 columns
Columns:    ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_ty

## Step 3: Build and Export `history_small.parquet` (Last 200 Days per Item)

### What we are doing in this cell
We are filtering `CA_1_clean.parquet` to strictly exclude the untouched test period (`date <= 2016-04-24`). Then, we sort by `item_id` and `date`, keep only the **last 200 days per item**, select only the columns required to build future forecasting features (`item_id`, `dept_id`, `cat_id`, `store_id`, `date`, `sales`, `sell_price`), and save the result as `history_small.parquet` inside `/content/backend_handover/`.

### Why we need this step
The full `CA_1_clean.parquet` file has 4.78 million rows, which is too heavy for a fast web backend. Keeping only the last 200 days per item gives us more than enough history to calculate 7-day and 28-day lags, rolling averages, and price changes while keeping the file lightweight.

### What problem this step solves
It dramatically shrinks the backend data footprint (from 4.78M rows down to ~609K rows) and strictly prevents test-data leakage by cutting off all dates after `2016-04-24`.

### What output we should expect
The shape of `history_small.parquet`, the number of unique items (3,049), the exact date range (ending on `2016-04-24`), missing value count, and the saved file size in MB.

In [ ]:
# 1. Ensure 'date' is in datetime format
ca1_clean["date"] = pd.to_datetime(ca1_clean["date"])

# 2. Strictly exclude the untouched test period (2016-04-25 to 2016-05-22)
# Validation period ends on 2016-04-24
CUTOFF_DATE = "2016-04-24"
history_no_test = ca1_clean[ca1_clean["date"] <= CUTOFF_DATE].copy()

# 3. Sort by item_id and date, then keep only the last 200 days per item
history_no_test = history_no_test.sort_values(["item_id", "date"])
history_small = history_no_test.groupby("item_id", sort=False).tail(200).copy()

# 4. Keep only the columns required to construct future forecasting features
# (item_id, dept_id for one-hot dept features, sales for lags/rolling, sell_price for price_change)
history_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "date",
    "sales",
    "sell_price"
]
history_small = history_small[history_cols].reset_index(drop=True)

# 5. Save history_small.parquet to our local backend_handover folder
history_small_path = os.path.join(HANDOVER_DIR, "history_small.parquet")
history_small.to_parquet(history_small_path, index=False)

# 6. Verify the exported file
size_mb = os.path.getsize(history_small_path) / (1024 * 1024)

print("--- history_small.parquet Summary ---")
print(f"Shape:        {history_small.shape[0]:,} rows, {history_small.shape[1]} columns")
print(f"Unique Items: {history_small['item_id'].nunique():,}")
print(f"Date Range:   {history_small['date'].min()} to {history_small['date'].max()}")
print(f"Missing Vals: {history_small.isnull().sum().sum()}")
print(f"[SAVED]       {history_small_path} ({size_mb:.2f} MB)")

print("\nPreview (First 5 rows):")
print(history_small.head())

--- history_small.parquet Summary ---
Shape:        609,372 rows, 7 columns
Unique Items: 3,049
Date Range:   2015-10-08 00:00:00 to 2016-04-24 00:00:00
Missing Vals: 0
[SAVED]       /content/backend_handover/history_small.parquet (0.40 MB)

Preview (First 5 rows):
       item_id  dept_id cat_id store_id       date  sales  sell_price
0  FOODS_1_001  FOODS_1  FOODS     CA_1 2015-10-08      0        2.24
1  FOODS_1_001  FOODS_1  FOODS     CA_1 2015-10-09      0        2.24
2  FOODS_1_001  FOODS_1  FOODS     CA_1 2015-10-10      1        2.24
3  FOODS_1_001  FOODS_1  FOODS     CA_1 2015-10-11      4        2.24
4  FOODS_1_001  FOODS_1  FOODS     CA_1 2015-10-12      1        2.24


## Step 4: Build and Export `calendar_small.csv`

### What we are doing in this cell
We are selecting the required calendar fields from `calendar.csv`—`date`, `weekday`, `wday`, `month`, `year`, `event_name_1`, `event_type_1`, and `snap_CA`—while preserving their exact source column names and values. We then save this lightweight table as `calendar_small.csv` inside `/content/backend_handover/`.

### Why we need this step
When our backend generates future forecasts (for example, predicting the next 7 or 28 days), it needs to know the upcoming weekday, month, year, holidays/events, and California SNAP days (`snap_CA`). We do not need Texas or Wisconsin SNAP columns (`snap_TX`, `snap_WI`) or unused M5 IDs.

### What problem this step solves
It creates a clean, compact calendar reference file that covers both our 200-day history and future forecast dates (`2011-01-29` to `2016-06-19`) without changing any original M5 column names or values.

### What output we should expect
The shape, columns, date range, and file size of `calendar_small.csv`, followed by a preview of the first 5 rows.

In [ ]:
# 1. Select only the required calendar columns for Store CA_1
# Preserving exact source column names and values from calendar.csv
calendar_cols = [
    "date",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA"
]

calendar_small = calendar_df[calendar_cols].copy()

# Ensure dates are sorted chronologically
calendar_small = calendar_small.sort_values("date").reset_index(drop=True)

# 2. Save calendar_small.csv to /content/backend_handover/
calendar_small_path = os.path.join(HANDOVER_DIR, "calendar_small.csv")
calendar_small.to_csv(calendar_small_path, index=False)

# 3. Verify the exported file
size_kb = os.path.getsize(calendar_small_path) / 1024

print("--- calendar_small.csv Summary ---")
print(f"Shape:      {calendar_small.shape[0]:,} rows, {calendar_small.shape[1]} columns")
print(f"Columns:    {list(calendar_small.columns)}")
print(f"Date Range: {calendar_small['date'].min()} to {calendar_small['date'].max()}")
print(f"[SAVED]     {calendar_small_path} ({size_kb:.2f} KB)")

print("\nPreview (First 5 rows):")
print(calendar_small.head())

--- calendar_small.csv Summary ---
Shape:      1,969 rows, 8 columns
Columns:    ['date', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA']
Date Range: 2011-01-29 to 2016-06-19
[SAVED]     /content/backend_handover/calendar_small.csv (65.44 KB)

Preview (First 5 rows):
         date    weekday  wday  month  year event_name_1 event_type_1  snap_CA
0  2011-01-29   Saturday     1      1  2011          NaN          NaN        0
1  2011-01-30     Sunday     2      1  2011          NaN          NaN        0
2  2011-01-31     Monday     3      1  2011          NaN          NaN        0
3  2011-02-01    Tuesday     4      2  2011          NaN          NaN        1
4  2011-02-02  Wednesday     5      2  2011          NaN          NaN        1


## Step 5: Verify Exact Notebook 03 Feature Formulas Before Writing `features.py`

### What we are doing in this cell
We are loading `ca1_h7_val.parquet` and `ca1_h28_val.parquet` and comparing their exact values for one sample item (`FOODS_1_001`) against our raw `history_small` data. Specifically, we check:
- How `lag_7`, `rolling_mean_7`, `lag_28`, and `rolling_mean_28` were calculated
- How `price_change` was calculated (e.g., difference vs. percentage change)
- How `is_weekend` and `has_event` were defined

### Why we need this step
In `features.py`, we must reproduce the **exact** feature engineering logic from Notebook 03 without guessing or simplifying anything. Even a tiny mismatch (like shifting a rolling mean by 1 day instead of 7 days) would cause wrong forecasts in production.

### What problem this step solves
It gives us 100% mathematical certainty on every feature formula before we write `features.py`.

### What output we should expect
A side-by-side comparison showing the exact values stored in `ca1_h7_val.parquet` and `ca1_h28_val.parquet` alongside our manual calculations from `history_small.parquet`.

In [ ]:
# 1. Load Notebook 03 validation feature files for comparison
val_h7 = pd.read_parquet("/content/ca1_h7_val.parquet")
val_h28 = pd.read_parquet("/content/ca1_h28_val.parquet")

# 2. Pick a sample item and check its first 3 validation rows
sample_item = "FOODS_1_001"

h7_sample = val_h7[val_h7["item_id"] == sample_item].sort_values("date").head(3)
h28_sample = val_h28[val_h28["item_id"] == sample_item].sort_values("date").head(3)

print(f"--- Notebook 03 Values for {sample_item} (H7) ---")
print(h7_sample[["date", "sales", "lag_7", "rolling_mean_7", "wday", "is_weekend", "sell_price", "price_change"]])

print(f"\n--- Notebook 03 Values for {sample_item} (H28) ---")
print(h28_sample[["date", "sales", "lag_28", "rolling_mean_28"]])

# 3. Reproduce lag, rolling mean, and price_change on history_small for the same item
hist_item = history_small[history_small["item_id"] == sample_item].sort_values("date").copy()

# Check lag_7 and rolling_mean_7 (shifted by 7 days so future actual sales are NEVER used)
hist_item["calc_lag_7"] = hist_item["sales"].shift(7)
hist_item["calc_roll_7"] = hist_item["sales"].shift(7).rolling(7).mean()

# Check lag_28 and rolling_mean_28 (shifted by 28 days)
hist_item["calc_lag_28"] = hist_item["sales"].shift(28)
hist_item["calc_roll_28"] = hist_item["sales"].shift(28).rolling(28).mean()

# Check price_change (difference vs percentage change)
hist_item["calc_price_diff"] = hist_item["sell_price"].diff()
hist_item["calc_price_pct"] = hist_item["sell_price"].pct_change()

# Filter to the exact same 3 dates (2016-03-28 to 2016-03-30)
check_dates = h7_sample["date"].values
calc_check = hist_item[hist_item["date"].isin(check_dates)]

print(f"\n--- Our Manual Calculation from history_small ({sample_item}) ---")
print(calc_check[["date", "sales", "calc_lag_7", "calc_roll_7", "calc_lag_28", "calc_roll_28"]])

# 4. Also check a row in val_h7 where price_change != 0 to confirm exact price_change formula
nonzero_price = val_h7[val_h7["price_change"] != 0].head(3)
print("\n--- Sample Rows Where price_change != 0 (in val_h7) ---")
print(nonzero_price[["item_id", "date", "sell_price", "price_change"]])

--- Notebook 03 Values for FOODS_1_001 (H7) ---
        date  sales  lag_7  rolling_mean_7  wday  is_weekend  sell_price  \
0 2016-03-28      2    0.0        1.000000     3           0        2.24   
1 2016-03-29      1    2.0        1.285714     4           0        2.24   
2 2016-03-30      1    2.0        1.571429     5           0        2.24   

   price_change  
0           0.0  
1           0.0  
2           0.0  

--- Notebook 03 Values for FOODS_1_001 (H28) ---
        date  sales  lag_28  rolling_mean_28
0 2016-03-28      2     1.0         0.750000
1 2016-03-29      1     0.0         0.678571
2 2016-03-30      1     1.0         0.714286

--- Our Manual Calculation from history_small (FOODS_1_001) ---
          date  sales  calc_lag_7  calc_roll_7  calc_lag_28  calc_roll_28
172 2016-03-28      2         0.0     1.000000          1.0      0.750000
173 2016-03-29      1         2.0     1.285714          0.0      0.678571
174 2016-03-30      1         2.0     1.571429          1.

## Step 6: Write and Verify `features.py` (Exact Notebook 03 Feature Engineering)

### What we are doing in this cell
We are creating a standalone Python module called `features.py` inside `/content/backend_handover/`. It contains `build_features()`, which reproduces the exact 27 features and exact column order from `feature_lists.json` for both **H7** (`horizon=7`) and **H28** (`horizon=28`) LightGBM models—without ever using future actual sales.

After saving `features.py`, we import it and verify its generated feature values against `ca1_h7_val.parquet` and `ca1_h28_val.parquet` for sample items (`FOODS_1_001` and `FOODS_1_098`).

### Why we need this step
When our backend generates new forecasts on demand, it cannot rely on notebook cells. It needs a clean, tested Python file (`features.py`) that transforms `history_small.parquet` + `calendar_small.csv` into the exact feature matrix LightGBM expects.

### What problem this step solves
It guarantees that backend features match training/validation features in both **values** and **column order**, preventing silent prediction errors or data leakage.

### What output we should expect
Confirmation that `features.py` was saved and imported, exact column order verification (`[PASS]`), and maximum numerical difference (`0.000000`) when compared against Notebook 03's validation features.

In [ ]:
import sys
import importlib

# 1. Define the standalone features.py module code
features_module_code = '''"""
DemandLens — Feature Engineering Module (Store CA_1)
Reproduces exact Notebook 03 feature engineering for H7 and H28 LightGBM models.
"""

import numpy as np
import pandas as pd


EVENT_TYPES = ["Cultural", "National", "Religious", "Sporting", "None"]
DEPARTMENTS = [
    "FOODS_1", "FOODS_2", "FOODS_3",
    "HOBBIES_1", "HOBBIES_2",
    "HOUSEHOLD_1", "HOUSEHOLD_2"
]


def build_features(history_df, calendar_df, feature_lists, horizon=7, future_dates=None):
    """
    Builds the exact 27-column feature matrix required by LightGBM (H7 or H28).

    Parameters:
    - history_df: DataFrame with columns ['item_id', 'dept_id', 'date', 'sales', 'sell_price']
    - calendar_df: DataFrame from calendar_small.csv
    - feature_lists: dict loaded from feature_lists.json
    - horizon: 7 (for H7 model) or 28 (for H28 model)
    - future_dates: optional list/DatetimeIndex of future dates to forecast
                    (uses NaN for future sales and last known sell_price, so no future sales are leaked)
    """
    if horizon not in (7, 28):
        raise ValueError("horizon must be 7 or 28.")

    df = history_df[["item_id", "dept_id", "date", "sales", "sell_price"]].copy()
    df["date"] = pd.to_datetime(df["date"])

    # If future_dates are requested, append skeleton rows with sales=NaN and last known price
    if future_dates is not None and len(future_dates) > 0:
        future_dates = pd.to_datetime(future_dates)
        last_rows = (
            df.sort_values(["item_id", "date"])
            .groupby("item_id", as_index=False)
            .last()[["item_id", "dept_id", "sell_price"]]
        )
        future_frames = []
        for fdate in future_dates:
            temp = last_rows.copy()
            temp["date"] = fdate
            temp["sales"] = np.nan  # Never use future actual sales
            future_frames.append(temp)
        df = pd.concat([df] + future_frames, ignore_index=True)

    # Sort chronologically per item before computing lags and rolling means
    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)

    # 1. Lag and rolling mean features (strictly shifted by 'horizon' days)
    grouped_sales = df.groupby("item_id", sort=False)["sales"]
    if horizon == 7:
        df["lag_7"] = grouped_sales.shift(7)
        df["rolling_mean_7"] = grouped_sales.transform(lambda s: s.shift(7).rolling(7).mean())
        expected_cols = feature_lists["features_7"]
    else:
        df["lag_28"] = grouped_sales.shift(28)
        df["rolling_mean_28"] = grouped_sales.transform(lambda s: s.shift(28).rolling(28).mean())
        expected_cols = feature_lists["features_28"]

    # 2. Weekly price change (sell_price minus sell_price 7 days ago)
    df["price_change"] = df.groupby("item_id", sort=False)["sell_price"].transform(
        lambda s: s - s.shift(7)
    ).fillna(0.0)

    # 3. Merge calendar features (wday, month, year, event_type_1, snap_CA)
    cal = calendar_df[["date", "wday", "month", "year", "event_type_1", "snap_CA"]].copy()
    cal["date"] = pd.to_datetime(cal["date"])
    df = df.merge(cal, on="date", how="left")

    # Rename snap_CA -> snap to match model feature name
    df["snap"] = df["snap_CA"].fillna(0).astype(int)

    # Weekend flag (in M5, wday 1 = Saturday, 2 = Sunday)
    df["is_weekend"] = df["wday"].isin([1, 2]).astype(int)

    # Event flags and one-hot columns
    df["has_event"] = df["event_type_1"].notna().astype(int)
    event_clean = df["event_type_1"].fillna("None")
    for ev in EVENT_TYPES:
        df[f"ev_{ev}"] = (event_clean == ev).astype(int)

    # 4. Department one-hot columns
    for dept in DEPARTMENTS:
        df[f"dept_{dept}"] = (df["dept_id"] == dept).astype(int)

    # 5. Cyclical time features
    df["wday_sin"] = np.sin(2 * np.pi * df["wday"] / 7)
    df["wday_cos"] = np.cos(2 * np.pi * df["wday"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # 6. Set item_id to exact categorical list from feature_lists.json
    df["item_id"] = pd.Categorical(
        df["item_id"],
        categories=feature_lists["item_categories"]
    )

    # If future_dates were requested, return only those future rows
    if future_dates is not None and len(future_dates) > 0:
        df = df[df["date"].isin(future_dates)].copy()

    # Keep date alongside the exact ordered feature columns for easy tracking
    return df[["date"] + expected_cols].reset_index(drop=True)
'''

# 2. Save features.py to /content/backend_handover/
features_py_path = os.path.join(HANDOVER_DIR, "features.py")
with open(features_py_path, "w") as f:
    f.write(features_module_code)

print(f"[SAVED] {features_py_path} ({os.path.getsize(features_py_path) / 1024:.2f} KB)")

# 3. Import build_features directly from features.py
if HANDOVER_DIR not in sys.path:
    sys.path.insert(0, HANDOVER_DIR)

import features
importlib.reload(features)
from features import build_features

print("[OK] Successfully imported 'build_features' from features.py!\n")

# 4. Verify against Notebook 03 (ca1_h7_val.parquet & ca1_h28_val.parquet) on sample items
test_items = ["FOODS_1_001", "FOODS_1_098"]
sample_hist = history_small[history_small["item_id"].isin(test_items)].copy()

gen_h7 = build_features(sample_hist, calendar_small, feature_lists, horizon=7)
gen_h28 = build_features(sample_hist, calendar_small, feature_lists, horizon=28)

# Check exact column order
h7_order_ok = list(gen_h7.columns[1:]) == feature_lists["features_7"]
h28_order_ok = list(gen_h28.columns[1:]) == feature_lists["features_28"]
print(f"[{'PASS' if h7_order_ok else 'FAIL'}] H7 Feature Column Order Matches feature_lists.json")
print(f"[{'PASS' if h28_order_ok else 'FAIL'}] H28 Feature Column Order Matches feature_lists.json")

# Compare generated values vs val_h7 on validation dates (2016-03-28 to 2016-04-24)
val_h7_sub = val_h7[val_h7["item_id"].isin(test_items)].sort_values(["item_id", "date"]).reset_index(drop=True)
gen_h7_sub = gen_h7[gen_h7["date"].isin(val_h7_sub["date"])].sort_values(["item_id", "date"]).reset_index(drop=True)

compare_cols = ["lag_7", "rolling_mean_7", "wday", "is_weekend", "snap", "has_event", "sell_price", "price_change"]
max_diff = (val_h7_sub[compare_cols].astype(float) - gen_h7_sub[compare_cols].astype(float)).abs().max().max()
print(f"[{'PASS' if max_diff < 1e-6 else 'FAIL'}] H7 Max Numerical Difference vs Notebook 03: {max_diff:.6f}")

# Test future feature generation (7 days after 2016-04-24, without using future actual sales)
future_7_dates = pd.date_range("2016-04-25", periods=7)
future_h7 = build_features(sample_hist, calendar_small, feature_lists, horizon=7, future_dates=future_7_dates)
print(f"[{'PASS' if len(future_h7) == len(test_items) * 7 and future_h7['lag_7'].isna().sum() == 0 else 'FAIL'}] Future 7-Day Feature Generation: {len(future_h7)} rows, 0 missing lags")

[SAVED] /content/backend_handover/features.py (4.37 KB)
[OK] Successfully imported 'build_features' from features.py!

[PASS] H7 Feature Column Order Matches feature_lists.json
[PASS] H28 Feature Column Order Matches feature_lists.json
[PASS] H7 Max Numerical Difference vs Notebook 03: 0.000000
[FAIL] Future 7-Day Feature Generation: 21343 rows, 0 missing lags


## Step 6 (Fix): Fix Categorical `groupby` Bug in `features.py` and Re-Verify

### What we are doing in this cell
We are updating `features.py` so that `item_id` is temporarily converted to a plain string (`astype(str)`) during grouping and future-row creation, and converted back to the exact `pd.Categorical` list from `feature_lists.json` at the end. Then we re-run all 4 verification tests.

### Why we need this step
When `history_df` is filtered to a subset of items (for example, 1 or 2 items in a backend API request), pandas `groupby()` on a `Categorical` column includes all 3,049 unused categories by default. That caused 21,343 rows (`3,049 × 7`) to be generated instead of 14 rows (`2 × 7`).

### What problem this step solves
It ensures that requesting forecasts for 1 item returns rows **only** for that 1 item, keeping backend predictions fast and accurate.

### What output we should expect
All 4 verification checks showing `[PASS]`, including `Future 7-Day Feature Generation: 14 rows, 0 missing lags`.

In [ ]:
import sys
import importlib

# 1. Updated features.py with clean string-based grouping (fixes unused Categorical rows bug)
features_module_code = '''"""
DemandLens — Feature Engineering Module (Store CA_1)
Reproduces exact Notebook 03 feature engineering for H7 and H28 LightGBM models.
"""

import numpy as np
import pandas as pd


EVENT_TYPES = ["Cultural", "National", "Religious", "Sporting", "None"]
DEPARTMENTS = [
    "FOODS_1", "FOODS_2", "FOODS_3",
    "HOBBIES_1", "HOBBIES_2",
    "HOUSEHOLD_1", "HOUSEHOLD_2"
]


def build_features(history_df, calendar_df, feature_lists, horizon=7, future_dates=None):
    """
    Builds the exact 27-column feature matrix required by LightGBM (H7 or H28).
    """
    if horizon not in (7, 28):
        raise ValueError("horizon must be 7 or 28.")

    df = history_df[["item_id", "dept_id", "date", "sales", "sell_price"]].copy()

    # Convert item_id to plain string during grouping so unused categories are never created
    df["item_id"] = df["item_id"].astype(str)
    df["date"] = pd.to_datetime(df["date"])

    # If future_dates are requested, append skeleton rows with sales=NaN and last known price
    if future_dates is not None and len(future_dates) > 0:
        future_dates = pd.to_datetime(future_dates)
        last_rows = (
            df.sort_values(["item_id", "date"])
            .groupby("item_id", as_index=False)
            .last()[["item_id", "dept_id", "sell_price"]]
        )
        future_frames = []
        for fdate in future_dates:
            temp = last_rows.copy()
            temp["date"] = fdate
            temp["sales"] = np.nan  # Never use future actual sales
            future_frames.append(temp)
        df = pd.concat([df] + future_frames, ignore_index=True)

    # Sort chronologically per item before computing lags and rolling means
    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)

    # 1. Lag and rolling mean features (strictly shifted by 'horizon' days)
    grouped_sales = df.groupby("item_id", sort=False)["sales"]
    if horizon == 7:
        df["lag_7"] = grouped_sales.shift(7)
        df["rolling_mean_7"] = grouped_sales.transform(lambda s: s.shift(7).rolling(7).mean())
        expected_cols = feature_lists["features_7"]
    else:
        df["lag_28"] = grouped_sales.shift(28)
        df["rolling_mean_28"] = grouped_sales.transform(lambda s: s.shift(28).rolling(28).mean())
        expected_cols = feature_lists["features_28"]

    # 2. Weekly price change (sell_price minus sell_price 7 days ago)
    df["price_change"] = df.groupby("item_id", sort=False)["sell_price"].transform(
        lambda s: s - s.shift(7)
    ).fillna(0.0)

    # 3. Merge calendar features (wday, month, year, event_type_1, snap_CA)
    cal = calendar_df[["date", "wday", "month", "year", "event_type_1", "snap_CA"]].copy()
    cal["date"] = pd.to_datetime(cal["date"])
    df = df.merge(cal, on="date", how="left")

    # Rename snap_CA -> snap to match model feature name
    df["snap"] = df["snap_CA"].fillna(0).astype(int)

    # Weekend flag (in M5, wday 1 = Saturday, 2 = Sunday)
    df["is_weekend"] = df["wday"].isin([1, 2]).astype(int)

    # Event flags and one-hot columns
    df["has_event"] = df["event_type_1"].notna().astype(int)
    event_clean = df["event_type_1"].fillna("None")
    for ev in EVENT_TYPES:
        df[f"ev_{ev}"] = (event_clean == ev).astype(int)

    # 4. Department one-hot columns
    for dept in DEPARTMENTS:
        df[f"dept_{dept}"] = (df["dept_id"] == dept).astype(int)

    # 5. Cyclical time features
    df["wday_sin"] = np.sin(2 * np.pi * df["wday"] / 7)
    df["wday_cos"] = np.cos(2 * np.pi * df["wday"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # 6. Set item_id to exact categorical list from feature_lists.json
    df["item_id"] = pd.Categorical(
        df["item_id"],
        categories=feature_lists["item_categories"]
    )

    # If future_dates were requested, return only those future rows
    if future_dates is not None and len(future_dates) > 0:
        df = df[df["date"].isin(future_dates)].copy()

    # Keep date alongside the exact ordered feature columns for easy tracking
    return df[["date"] + expected_cols].reset_index(drop=True)
'''

# 2. Overwrite features.py with the fixed version
features_py_path = os.path.join(HANDOVER_DIR, "features.py")
with open(features_py_path, "w") as f:
    f.write(features_module_code)

# 3. Reload features.py and re-run all 4 verification tests
if HANDOVER_DIR not in sys.path:
    sys.path.insert(0, HANDOVER_DIR)

import features
importlib.reload(features)
from features import build_features

print(f"[SAVED] Updated {features_py_path}")
print("[OK] Reloaded 'build_features' from features.py!\n")

test_items = ["FOODS_1_001", "FOODS_1_098"]
sample_hist = history_small[history_small["item_id"].isin(test_items)].copy()

gen_h7 = build_features(sample_hist, calendar_small, feature_lists, horizon=7)
gen_h28 = build_features(sample_hist, calendar_small, feature_lists, horizon=28)

h7_order_ok = list(gen_h7.columns[1:]) == feature_lists["features_7"]
h28_order_ok = list(gen_h28.columns[1:]) == feature_lists["features_28"]
print(f"[{'PASS' if h7_order_ok else 'FAIL'}] H7 Feature Column Order Matches feature_lists.json")
print(f"[{'PASS' if h28_order_ok else 'FAIL'}] H28 Feature Column Order Matches feature_lists.json")

val_h7_sub = val_h7[val_h7["item_id"].isin(test_items)].sort_values(["item_id", "date"]).reset_index(drop=True)
gen_h7_sub = gen_h7[gen_h7["date"].isin(val_h7_sub["date"])].sort_values(["item_id", "date"]).reset_index(drop=True)

compare_cols = ["lag_7", "rolling_mean_7", "wday", "is_weekend", "snap", "has_event", "sell_price", "price_change"]
max_diff = (val_h7_sub[compare_cols].astype(float) - gen_h7_sub[compare_cols].astype(float)).abs().max().max()
print(f"[{'PASS' if max_diff < 1e-6 else 'FAIL'}] H7 Max Numerical Difference vs Notebook 03: {max_diff:.6f}")

future_7_dates = pd.date_range("2016-04-25", periods=7)
future_h7 = build_features(sample_hist, calendar_small, feature_lists, horizon=7, future_dates=future_7_dates)
print(f"[{'PASS' if len(future_h7) == len(test_items) * 7 and future_h7['lag_7'].isna().sum() == 0 else 'FAIL'}] Future 7-Day Feature Generation: {len(future_h7)} rows, {future_h7['lag_7'].isna().sum()} missing lags")

[SAVED] Updated /content/backend_handover/features.py
[OK] Reloaded 'build_features' from features.py!

[PASS] H7 Feature Column Order Matches feature_lists.json
[PASS] H28 Feature Column Order Matches feature_lists.json
[PASS] H7 Max Numerical Difference vs Notebook 03: 0.000000
[PASS] Future 7-Day Feature Generation: 14 rows, 0 missing lags


## Step 7: Final Backend Handover Organization & End-to-End Verification

### What we are doing in this cell
We are gathering all 11 required backend files into `/content/backend_handover/` (without overwriting our newly created files) and running a complete end-to-end verification:
1. Confirming all 11 required files exist and checking their file sizes
2. Checking columns, row counts, and data types of every tabular file
3. Loading both LightGBM models (`H7` and `H28`) directly from the handover folder
4. Running an end-to-end future forecast test (`features.py` -> `model_h7` and `model_h28`) and confirming exact feature order
5. Verifying **zero test-data leakage** in `history_small.parquet` (`max date <= 2016-04-24`)
6. Printing the final **Backend Handover Checklist** (noting `anomalies_test.parquet` is deferred until final test evaluation)

### Why we need this step
Before moving to the Flask backend or final evaluation stage, we must prove that the `backend_handover` folder is 100% self-contained, leak-free, and ready to run independently.

### What problem this step solves
It catches any missing file, broken model load, or feature mismatch before backend development begins.

### What output we should expect
A complete verification report showing `[PASS]` for model loading, feature generation, prediction, and zero test-data leakage, followed by the final 12-item Backend Handover Checklist.

In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

HANDOVER_DIR = "/content/backend_handover"
os.makedirs(HANDOVER_DIR, exist_ok=True)

# 1. Copy remaining existing backend files from /content/ into /content/backend_handover/
# (Only copy if not already inside HANDOVER_DIR so we do not overwrite verified files)
files_to_copy = [
    "lightgbm_h7.joblib",
    "lightgbm_h28.joblib",
    "feature_lists.json",
    "anomalies_validation.parquet",
    "item_groups.csv",
    "group_thresholds.csv",
    "item_error_spread.csv",
    "reorder.py"
]

for fname in files_to_copy:
    src = os.path.join("/content", fname)
    dst = os.path.join(HANDOVER_DIR, fname)
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy2(src, dst)

# 2. Define the 11 required files for now (anomalies_test.parquet is deferred until final test evaluation)
required_files = [
    "lightgbm_h7.joblib",
    "lightgbm_h28.joblib",
    "feature_lists.json",
    "features.py",
    "history_small.parquet",
    "calendar_small.csv",
    "anomalies_validation.parquet",
    "item_groups.csv",
    "group_thresholds.csv",
    "item_error_spread.csv",
    "reorder.py"
]

print("=== 1. FILE EXISTENCE & SIZE CHECK ===")
all_present = True
for fname in required_files:
    fpath = os.path.join(HANDOVER_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        size_str = f"{size_kb/1024:.2f} MB" if size_kb >= 1024 else f"{size_kb:.2f} KB"
        print(f"[PASS] {fname:<30} ({size_str})")
    else:
        print(f"[FAIL] {fname:<30} MISSING!")
        all_present = False

print("[DEFERRED] anomalies_test.parquet     (Will be generated after final test evaluation)")

# 3. Check columns, row counts, and data types for all tabular files
print("\n=== 2. TABULAR DATA SCHEMA CHECK ===")
tabular_files = [
    "history_small.parquet",
    "calendar_small.csv",
    "anomalies_validation.parquet",
    "item_groups.csv",
    "group_thresholds.csv",
    "item_error_spread.csv"
]

for fname in tabular_files:
    fpath = os.path.join(HANDOVER_DIR, fname)
    df_tmp = pd.read_parquet(fpath) if fname.endswith(".parquet") else pd.read_csv(fpath)
    print(f"\nFile: {fname} -> {df_tmp.shape[0]:,} rows x {df_tmp.shape[1]} cols")
    print("Columns & Dtypes:", {col: str(dtype) for col, dtype in df_tmp.dtypes.items()})

# 4. End-to-End Test: Load models + generate future features + predict
print("\n=== 3. END-TO-END MODEL & FEATURE VERIFICATION ===")
m_h7 = joblib.load(os.path.join(HANDOVER_DIR, "lightgbm_h7.joblib"))
m_h28 = joblib.load(os.path.join(HANDOVER_DIR, "lightgbm_h28.joblib"))
with open(os.path.join(HANDOVER_DIR, "feature_lists.json"), "r") as f:
    flists = json.load(f)

hist_check = pd.read_parquet(os.path.join(HANDOVER_DIR, "history_small.parquet"))
cal_check = pd.read_csv(os.path.join(HANDOVER_DIR, "calendar_small.csv"))

# Check zero test-data leakage
max_hist_date = pd.to_datetime(hist_check["date"]).max()
no_leakage = max_hist_date <= pd.Timestamp("2016-04-24")
print(f"[{'PASS' if no_leakage else 'FAIL'}] Zero Test-Data Leakage (history_small max date = {max_hist_date.date()})")

# Generate future 7-day and 28-day features for 1 sample item and run predictions
sample_df = hist_check[hist_check["item_id"] == "FOODS_1_001"].copy()

f7 = build_features(sample_df, cal_check, flists, horizon=7, future_dates=pd.date_range("2016-04-25", periods=7))
f28 = build_features(sample_df, cal_check, flists, horizon=28, future_dates=pd.date_range("2016-04-25", periods=28))

# Verify exact feature order against model expectations
h7_order_match = list(f7.columns[1:]) == list(m_h7.feature_name_)
h28_order_match = list(f28.columns[1:]) == list(m_h28.feature_name_)
print(f"[{'PASS' if h7_order_match else 'FAIL'}] H7 Feature Order Matches Trained Model")
print(f"[{'PASS' if h28_order_match else 'FAIL'}] H28 Feature Order Matches Trained Model")

# Run predictions
pred_h7 = np.clip(m_h7.predict(f7[flists["features_7"]]), 0, None)
pred_h28 = np.clip(m_h28.predict(f28[flists["features_28"]]), 0, None)

preds_ok = (len(pred_h7) == 7) and (len(pred_h28) == 28) and (not np.isnan(pred_h7).any()) and (not np.isnan(pred_h28).any())
print(f"[{'PASS' if preds_ok else 'FAIL'}] End-to-End Future Predictions (H7 mean={pred_h7.mean():.2f}, H28 mean={pred_h28.mean():.2f})")

print("\n=== FINAL HANDOVER STATUS ===")
if all_present and no_leakage and h7_order_match and h28_order_match and preds_ok:
    print("ALL CHECKS PASSED! Notebook 10 (Backend Data Export) is 100% complete.")
else:
    print("SOME CHECKS FAILED! Please inspect the report above.")

=== 1. FILE EXISTENCE & SIZE CHECK ===
[PASS] lightgbm_h7.joblib             (1.74 MB)
[PASS] lightgbm_h28.joblib            (8.33 MB)
[PASS] feature_lists.json             (62.78 KB)
[PASS] features.py                    (4.06 KB)
[PASS] history_small.parquet          (411.89 KB)
[PASS] calendar_small.csv             (65.44 KB)
[PASS] anomalies_validation.parquet   (248.85 KB)
[PASS] item_groups.csv                (307.73 KB)
[PASS] group_thresholds.csv           (0.27 KB)
[PASS] item_error_spread.csv          (221.30 KB)
[PASS] reorder.py                     (3.94 KB)
[DEFERRED] anomalies_test.parquet     (Will be generated after final test evaluation)

=== 2. TABULAR DATA SCHEMA CHECK ===

File: history_small.parquet -> 609,372 rows x 7 cols
Columns & Dtypes: {'item_id': 'category', 'dept_id': 'category', 'cat_id': 'category', 'store_id': 'category', 'date': 'datetime64[ns]', 'sales': 'int16', 'sell_price': 'float32'}

File: calendar_small.csv -> 1,969 rows x 8 cols
Columns & Dtypes